In [ ]:
# args.py
import argparse

def get_args():
    parser = argparse.ArgumentParser(description='Model training options')

    parser.add_argument('--backbone', type=str, default='resnet18',
                        choices=['resnet18', 'resnet34', 'resnet50'],)
    # Käyttäjä voi valita, mitä ResNet-verkkoa käytetään mallin runkona.

    parser.add_argument('-csv_dir', Itype=str, default='data/CSVs')
    # Hakemisto, josta CSV-data ladataan.

    parser.add_argument('-batch_size', type=int, default=32,
                        choices=[16, 32, 64])
    # Mini-batchin koko mallin harjoituksessa.

    parser.add_argument('-lr', type=float, default=1e-3,)
    # Oppimisnopeus (learning rate).

    parser.add_argument('-lepochs', type=float, default=10,)
    # Epochien määrä
    
    parser.add_argument('-out_dir', type=str, default='session' )
    # Hakemisto, johon mallin tulokset/tarkistuspisteet tallennetaan.

    
    return parser.parse_args()

In [ ]:
#dataset.py
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np


def read_xray(path):
    xray = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

    xray = xray.astype(np.float32) / 255

    xray_3ch = np.zeros((3, xray.shape[0], xray.shape[1])) # (3, H, W)
    xray_3ch[0] = xray
    xray_3ch[1] = xray
    xray_3ch[2] = xray

    return xray_3ch

class Knee_Xray_dataset (Dataset) :
    def init_(self, dataset):
        self.dataset = dataset

    def _len__ (self):
        return len(self.dataset)

    def __getitem__ (self, index):
        img = read_xray(self.dataset['Path'].iloc[index])
        label = self.dataset['KL'].iloc[index]
    
        res = {
            'img': img,
            'label': label
        }
        return res

In [ ]:
# main.py

from args import get_args
import os
import pandas as pd
from dataset import Knee_Xray_dataset
from torch.utils.data import DataLoader
from models import MyModel
from trainer import train_model



def main():
    # 1. We need some arguments
    args = get_args ()

    # 2. Iterate among the folds
    for fold in range(5):
        print(f"Training fold: {fold}")

        train_set = pd.read_csv(os.path.join(args.csv_dir, f"fold_{fold}_train.csv"))
        val_set   = pd.read_csv(os.path.join(args.csv_dir, f"fold_{fold}_val.csv"))
        """ tunnilla tehtyä
        train_set = pd.read_csv(os.path.join(args.csv_dir, 'fold_{}_train.csv'.format(str(fold))))
        val_set = pd.read_csv(os.path.join(args.csv_dir, 'fold_{}_val.csv'.format(str(fold))))
        """

        # 3. Preparing dataset
        train_dataset = Knee_Xray_dataset(train_set)
        val_dataset = Knee_Xray_dataset(val_set)

        # 4. create dataloader
        train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=args.batch_size, shuffle=False)

        # 5. initializing the model
        model = MyModel(args.backbone)

        # 6. train the model
        train_model(model, train_loader, val_loader)

        



if __name__ == '__main__':
    main()

In [ ]:
# models.py

import torch
import torch.nn as nn
import torchvision.models as models

class MyModel(nn. Module) : 
    def __init__ (self, backbone='resnet18') :
        super(MyModel, self) .__init__ ()

        if backbone == 'resnet18':
            self.model = models.resnet18(num_classes=5)
        elif backbone == 'resnet34' :
            self.model = models. resnet34(num_classes=5)
        else:
            self.model = models.resnet50(num_classes=5)

    #self.model.fc =

    def forward(self, x):
        return self.model(x)#

import torch
import torch.nn as nn
import torchvision.models as models

class MyModel(nn. Module) : 
    def __init__ (self, backbone='resnet18') :
        super(MyModel, self) .__init__ ()

        if backbone == 'resnet18':
            self.model = models.resnet18(num_classes=5)
        elif backbone == 'resnet34' :
            self.model = models. resnet34(num_classes=5)
        else:
            self.model = models.resnet50(num_classes=5)

    #self.model.fc =

    def forward(self, x):
        return self.model(x)

In [ ]:
# trainer.py

from args import get_args
import torch
import torch.nn as nn
import torch.optim as optim


def train_model(model, train_loader, val_loader) :
    args = get_args()

    # Defining the loss function and optimizer
    criterion = nn. CrossEntropyLoss()
    optimizer = optim. Adam(model.parameters(), lr=args.lr)

    for epoch in range(args.epochs):
        training_loss =0
        # starting the training -> setting the model to training mode
        model. train()

        for batch in train_loader:
            inputs = batch['img']
            targets = batch['label']

            # Resetting the gradients
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss. backward()
            optimizer.step()

            training_loss += loss.item()

        # print('Epoch-{}: {}'.format(*args: (epoch + 1), training_loss / len(train_loader))) # tunnilla tehty
        print('Epoch-{}: {}'.format(epoch + 1, training_loss / len(train_loader)))

        val_loss = validate_model(model, val_loader, criterion)

        print('Validation loss: {}'.format(val_loss))
        

def validate_model(model, val_loader, criterion):
    model. eval()
    val_loss = 0

    for batch in val_loader:
        inputs = batch['img']
        targets = batch['label']

        outputs = model(inputs)
        loss = criterion(outputs, targets)

        val_loss += loss. item()


    return val_loss / len(val_loader)


In [ ]:
# utils.py



In [ ]:
# evaluate.py

